# QuantLab · Explorar el data lake
Cuaderno de exploración: consulta con **DuckDB** los ficheros **Parquet** que guarda el capturador (`python -m radar capturar`) y grafica lo capturado.

- Cada carpeta de `data/lake/` es una **vista** con su nombre (por ejemplo `cotizaciones`).
- Las horas se guardan en **UTC**; aquí se muestran también en UTC para no mezclar husos.
- Vuelve a ejecutar el cuaderno para ver las capturas nuevas: los Parquet se leen en cada consulta.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt

sys.path.insert(0, str(next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "proyecto" / "radar").exists()) / "proyecto"))
from radar.almacen import LakeParquet

lake = LakeParquet()
consultar = lake.consultar
plt.rcParams.update({'figure.figsize': (10, 3.5), 'axes.grid': True, 'grid.alpha': .25})
print('Lake:', lake.raiz, '| fuentes:', lake.fuentes())
print('Ficheros Parquet:', len(list(lake.raiz.rglob('*.parquet'))))

## 1. Qué hay guardado
Resumen por fuente y día: filas, primera y última hora (UTC).

In [ ]:
consultar('''
SELECT fecha, exchange, count(*) AS fotos,
       strftime(min(recibido_utc) AT TIME ZONE 'UTC', '%H:%M:%S') AS desde_utc,
       strftime(max(recibido_utc) AT TIME ZONE 'UTC', '%H:%M:%S') AS hasta_utc
FROM cotizaciones GROUP BY fecha, exchange ORDER BY fecha, exchange
''')

## 2. Estadísticos del spread y de la latencia
- **Spread absoluto** (€) y **relativo** (bps = spread / mid × 10.000).
- **Latencia**: recepción − solicitud (s).

In [ ]:
consultar('''
SELECT exchange, count(*) AS fotos,
       round(median(spread_eur), 2) AS spread_mediano_eur, round(max(spread_eur), 2) AS spread_max_eur,
       round(median(spread_bps), 4) AS spread_mediano_bps, round(quantile_cont(spread_bps, 0.9), 4) AS spread_p90_bps,
       round(median(latencia_s), 3) AS latencia_mediana_s, round(max(latencia_s), 3) AS latencia_max_s
FROM cotizaciones GROUP BY exchange
''')

## 3. Precio y spread en el tiempo

In [ ]:
serie = consultar('''
SELECT recibido_utc AT TIME ZONE 'UTC' AS hora_utc, bid, ask, mid, spread_eur, spread_bps, cantidad_bid, cantidad_ask
FROM cotizaciones WHERE exchange = 'kraken' ORDER BY recibido_utc
''').set_index('hora_utc')

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
serie[['bid', 'ask']].plot(ax=axes[0], drawstyle='steps-post', title='Kraken BTC/EUR · mejor bid y ask')
axes[0].set_ylabel('EUR')
serie['spread_eur'].plot(ax=axes[1], drawstyle='steps-post', color='tab:red', title='Spread absoluto')
axes[1].set_ylabel('EUR'); axes[1].set_xlabel('Hora (UTC)')
plt.tight_layout(); plt.show()

## 4. Las fotos con el spread más ancho
¿Qué pasaba en el mejor nivel cuando el spread se ensanchó?

In [ ]:
serie.sort_values('spread_eur', ascending=False).head(5)

## 5. Consulta libre
Escribe aquí cualquier SQL sobre las vistas del lake.

In [ ]:
consultar('SELECT * FROM cotizaciones ORDER BY recibido_utc DESC LIMIT 5')